# 00 Probe

Spec sections 13-14. One agent per model tier writes a bot (author call) and revises it once, with no teaching; a second independent agent of the same tier does the same; we cross-play the two.

**Scope: stub backend only.** No API key was used. Every number below exercises the pipeline (scores of perturbed anchor bots, fake token counts priced at the tier's list price) and says nothing about a real model. The same notebook runs unchanged once a real backend exists.

In [1]:
import json, os, sys
sys.path.insert(0, 'src')  # repo root is the working directory
os.environ.setdefault('PYTHONHASHSEED', '0')
from culture.analysis.probe import run_probe
OUT = 'runs/probe'
results = {m: run_probe('configs/probe.yaml', OUT, m) for m in ['claude-haiku-4-5', 'claude-sonnet-5']}

In [2]:
for m, r in results.items():
    print(m)
    for a, v in r['agents'].items():
        print(f"  {a}: self-play {v['selfplay']:.2f}, illegal {v['illegal_rate']:.3f}, {v['lines']} lines, H-group keywords {v['hgroup_keywords']}")
    print(f"  cross-play between the two independent agents: {r['crossplay_between_independent_agents']:.2f}")
    print(f"  conventions bag-of-words cosine between agents: {r['conventions_similarity_between_agents']:.2f}")
    for t, c in r['cost_per_call'].items():
        print(f"  {t}: {c['calls']} calls, {c['tokens_per_call']:.0f} tokens/call, ${c['usd_per_call']:.4f}/call (nominal, stub)")
    print('  gates:', r['gates'])

claude-haiku-4-5
  g0a0: self-play 0.00, illegal 0.000, 220 lines, H-group keywords {}
  g1a0: self-play 8.69, illegal 0.000, 220 lines, H-group keywords {}
  cross-play between the two independent agents: 0.00
  conventions bag-of-words cosine between agents: 0.71
  author: 2 calls, 3750 tokens/call, $0.0139/call (nominal, stub)
  revise: 2 calls, 7598 tokens/call, $0.0169/call (nominal, stub)
  teach: 1 calls, 3781 tokens/call, $0.0039/call (nominal, stub)
  gates: {'headroom_selfplay_below_20': True, 'cost_measured': True, 'null_alive_crossplay_well_below_selfplay': True}
claude-sonnet-5
  g0a0: self-play 0.00, illegal 0.000, 220 lines, H-group keywords {}
  g1a0: self-play 1.06, illegal 0.000, 220 lines, H-group keywords {}
  cross-play between the two independent agents: 0.10
  conventions bag-of-words cosine between agents: 0.84
  author: 2 calls, 3732 tokens/call, $0.0275/call (nominal, stub)
  revise: 2 calls, 8230 tokens/call, $0.0350/call (nominal, stub)
  teach: 1 calls, 381

## Gates (decided before running, spec section 14)

- **Headroom**: generation-one bots below 20 in self-play.
- **Cost**: tokens per revise and teach call measured; stage sizes derive from them.
- **Null alive**: independent agents' cross-play well below their self-play (here: below 0.8 of mean self-play).

With the stub, the null gate is expected to **fail**: stub bots are all variants of one rule-list template and share card-index semantics, so they are mutually compatible. That is a property of the stub, not evidence about LLM-written conventions.

In [3]:
print(json.dumps({m: r['gates'] for m, r in results.items()}, indent=1))

{
 "claude-haiku-4-5": {
  "headroom_selfplay_below_20": true,
  "cost_measured": true,
  "null_alive_crossplay_well_below_selfplay": true
 },
 "claude-sonnet-5": {
  "headroom_selfplay_below_20": true,
  "cost_measured": true,
  "null_alive_crossplay_well_below_selfplay": true
 }
}


## Innovation base rate (fix round 1, step 12)

epsilon = P(a revision beats its parent on held-out deals by more than a declared margin), per revise-context condition: **nothing** (no feedback traces, nothing received), **feedback_only** (own failure traces on the generation's feedback deals), **feedback_plus_received** (traces plus teaching messages and the group corpus). Outcomes are paired: candidate and parent on the same held-out evaluation deals, disjoint from the feedback deals. Beta(1, 1) prior, 95% equal-tailed credible interval.

**Stub only here**: the stub ignores the context (it mutates the rule list it finds), so the three rates should agree up to noise; this cell proves the estimator and the plumbing. With a real backend the same cell measures whether context changes epsilon.

In [4]:
from culture.analysis.innovation import run_innovation_probe
from culture.run.config import load_config
base = load_config('configs/probe.yaml', {'name': 'probe_innovation', 'population': {'groups': 1, 'agents_per_group': 3}, 'evaluation': {'selfplay_games': 40, 'anchor_games': 20, 'between_group_games': 0}, 'runner': {'generations': 6}})
MARGIN = 0.5  # declared before running: a revision must gain more than half a point on held-out deals
inno = run_innovation_probe(base, 'runs/probe_innovation', margin=MARGIN, seeds=(0, 1))
for c, e in inno['by_context'].items():
    print(f"{c:24s} n={e['n']:3d}  innovations={e['successes']:3d}  epsilon={e['mean']:.3f}  95% CrI [{e['lo']:.3f}, {e['hi']:.3f}]")

feedback_only            n= 30  innovations=  1  epsilon=0.062  95% CrI [0.008, 0.167]
feedback_plus_received   n= 30  innovations=  1  epsilon=0.062  95% CrI [0.008, 0.167]
nothing                  n= 30  innovations=  1  epsilon=0.062  95% CrI [0.008, 0.167]
